## Proyecto 2 Modelos de Machine Learning  y selección del mejor

El presente análisis se basa en un conjunto de datos proveniente del Sloan Digital Sky Survey (SDSS), que contiene 100,000 observaciones astronómicas. Cada registro incluye 17 variables descriptivas, como coordenadas angulares, filtros fotométricos en distintas bandas (ultravioleta, verde, rojo, infrarrojo cercano e infrarrojo), valores de desplazamiento al rojo (redshift) e identificadores técnicos de procesamiento, junto con una columna de clase que etiqueta cada objeto como estrella, galaxia o cuásar. El objetivo de este informe es explorar y modelar dichos datos para construir un sistema de clasificación robusto que, a partir de las propiedades espectrales y fotométricas, permita distinguir automáticamente entre estas tres clases de objetos celestes. A lo largo del documento se describirán las etapas de preparación de los datos, el análisis exploratorio, la selección de características y la evaluación de distintos algoritmos de aprendizaje supervisado, con el fin de obtener un clasificador preciso y útil para aplicaciones astronómicas.

A continuación, se presenta una descripción para cada una de las variables del conjunto de datos:

* obj_ID: Identificador único del objeto en el catálogo de imágenes del CAS.

* alpha: Ángulo de ascensión recta (coordenada ecuatorial, época J2000).

* delta: Ángulo de declinación (coordenada ecuatorial, época J2000).

* u: Flujo medido con el filtro ultravioleta del sistema fotométrico.

* g: Flujo medido con el filtro verde del sistema fotométrico.

* r: Flujo medido con el filtro rojo del sistema fotométrico.

* i: Flujo medido con el filtro de infrarrojo cercano del sistema fotométrico.

* z: Flujo medido con el filtro infrarrojo del sistema fotométrico.

* run_ID: Número de ejecución que identifica el barrido (scan) específico.

* rerun_ID: Número de reprocesamiento que indica cómo se procesó la imagen.

* cam_col: Columna de la cámara que identifica la línea de escaneo dentro de la ejecución.

* field_ID: Número de campo que identifica cada campo observado.

* spec_obj_ID: Identificador único para objetos espectroscópicos ópticos (mismo valor implica misma clase de salida).

* class: Clase del objeto celeste: galaxia, estrella o cuásar.

* redshift: Desplazamiento al rojo, basado en el aumento de la longitud de onda.

* plate: Identificador de la placa (plate) utilizada en el SDSS.

* MJD: Fecha Juliana Modificada, que indica cuándo se tomó el dato.

* fiber_ID: Identificador de la fibra óptica que dirigió la luz hacia el plano focal en cada observación.

**Nota metodológica.** Para este informe se construyó una muestra **balanceada** de 10 000 registros (50% STAR, 25% GALAXY, 25% QSO) y, siguiendo un enfoque de clasificación binaria, las categorías **GALAXY y QSO se agruparon en una única clase “extragaláctica”**. El problema final consiste por tanto en distinguir objetos estelares (STAR) de objetos extragalácticos (GALAXY + QSO), con clases perfectamente balanceadas (50%/50%).

In [ ]:
import pandas as pd

# Carga de datos
df = pd.read_csv('/content/star_classification.csv')

print("Original dataset shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
display(df.head())

df.describe()

In [ ]:
# Verificación de la distribución de las clases
print("\nClass distribution in the original dataset:")
display(df['class'].value_counts())

In [ ]:
# Proporción ORIGINAL de cada clase (solo de referencia)
proporcion_original = df['class'].value_counts(normalize=True)
print("\nProporción ORIGINAL por clase (referencia):")
display(proporcion_original)


In [ ]:
# --- DISEÑO DE MUESTREO BALANCEADO (LEVE) ---
# En lugar de conservar la proporción original (dominada por GALAXY), se diseña una
# muestra balanceada según la observación metodológica:
#   STAR = 50%, GALAXY = 25%, QSO = 25%
# Posteriormente GALAXY y QSO se combinan en una sola categoría "extragaláctica",
# lo que produce un problema binario perfectamente balanceado: STAR (50%) vs GAL+QSO (50%).

registros_muestreados = 10000

proporcion_objetivo = {
    'STAR':   0.50,
    'GALAXY': 0.25,
    'QSO':    0.25,
}

muestra_por_clase = {
    clase: int(round(prop * registros_muestreados))
    for clase, prop in proporcion_objetivo.items()
}

print("Diseño de muestreo OBJETIVO (balanceado):")
for clase, n in muestra_por_clase.items():
    print(f"  {clase:8s}: {n:5d}  ({proporcion_objetivo[clase]*100:.0f}%)")

print("\nDisponibles en el dataset original:")
display(df['class'].value_counts())


In [ ]:
# Nuevo DataFrame para los datos reducidos
df_reduced = pd.DataFrame()

# Muestreo de cada clase individual
for class_name, num_samples in muestra_por_clase.items():
    class_df = df[df['class'] == class_name]
    # Asegurarse que no exista una muestra mayor a la población disponible
    num_samples = min(num_samples, len(class_df))
    sampled_class_df = class_df.sample(n=num_samples, random_state=42)
    df_reduced = pd.concat([df_reduced, sampled_class_df])

# Alternar las muestras en el nuevo dataset
df_reduced = df_reduced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nTamaño del nuevo Dataset:", df_reduced.shape)
print("\nDistribución de las clases en el nuevo Dataset:")
display(df_reduced['class'].value_counts())
print("\nPrimeras 5 filas:")
display(df_reduced.head())

In [ ]:
# Creación de nuevo archivo CSV
df_reduced.to_csv('/content/star_classification_reduced.csv', index=False)

print("Nuevo Archivo guardado en '/content/star_classification_reduced.csv'")

## Carga de Librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             accuracy_score, f1_score, roc_curve)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import xgboost as xgb
import shap
# !pip install lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## CLASIFICACIÓN DE OBJETOS ASTRONÓMICOS

El análisis se realiza con las variables `alpha, delta, u, g, r, i, z, redshift`, ya que el resto (identificadores del objeto, del equipo utilizado, etc.) no aportan información discriminativa para la clasificación.

**Formulación del problema.** Tras el muestreo balanceado (50% STAR, 25% GALAXY, 25% QSO), se agrupan **GALAXY y QSO en una única categoría extragaláctica**. De este modo el problema se convierte en una clasificación **binaria balanceada**:

- `0 = STAR` (objeto estelar, redshift ≈ 0)
- `1 = GAL+QSO` (objeto extragaláctico, redshift > 0)

Antes de modelar se examina la correlación entre variables y se realiza una **prueba de multicolinealidad (VIF)** para decidir si es necesario eliminar variables redundantes.

In [ ]:
df = pd.read_csv('/content/star_classification_reduced.csv')

print("\nExploración Inicial")
print(f"Dimensiones: {df.shape}")
print(f"Distribución de clases (3 categorías):\n{df['class'].value_counts()}")

# --- AGRUPACIÓN DE CATEGORÍAS ---
# Se combinan GALAXY y QSO en una única categoría "extragaláctica".
# Variable objetivo binaria:
#   1 = GAL+QSO (objetos extragalácticos, redshift > 0)
#   0 = STAR    (objetos estelares de la Vía Láctea, redshift ~ 0)
df['target'] = (df['class'] != 'STAR').astype(int)

print("\nDistribución BINARIA tras agrupar GALAXY + QSO:")
print(df['target'].value_counts().rename({0: 'STAR (0)', 1: 'GAL+QSO (1)'}))

# Visualización: 3 categorías originales y la agrupación binaria
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='class', data=df, palette='viridis', ax=axes[0])
axes[0].set_title('Muestreo balanceado (3 categorías)')
axes[0].set_xlabel('Clase'); axes[0].set_ylabel('Conteo')

sns.countplot(x='target', data=df, palette='mako', ax=axes[1])
axes[1].set_title('Variable objetivo binaria (GAL+QSO agrupadas)')
axes[1].set_xticklabels(['STAR (0)', 'GAL+QSO (1)'])
axes[1].set_xlabel('Categoría'); axes[1].set_ylabel('Conteo')
plt.tight_layout(); plt.show()

feature_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
X = df[feature_cols].copy()
y = df['target'].copy()

# División para entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}")


In [ ]:
print("\nANÁLISIS DE CORRELACIÓN")
correlation_matrix = X.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlación de Variables')
plt.show()

Se observa que las variables u, g, r, i, z, tiene una alta correlación entre ellas y baja correlación con las otras varaibles.
Estas variables, u, g, r, i, z, están relacionadas dado que son filtros fotometrico de la misma imagen.

## Prueba de multicolinealidad (VIF)

Dado que la matriz de correlación reveló relaciones muy fuertes entre las bandas fotométricas `u, g, r, i, z`, se realiza una prueba formal de multicolinealidad mediante el **Factor de Inflación de la Varianza (VIF)**. Como regla práctica, un VIF > 5 indica multicolinealidad moderada y un VIF > 10, multicolinealidad severa. Adicionalmente se evalúa de forma empírica si esta redundancia afecta el desempeño predictivo, comparando un modelo lineal con todas las variables frente a otro con un conjunto reducido.

In [ ]:
print("PRUEBA DE MULTICOLINEALIDAD (VIF)\n")

X_vif = add_constant(X)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_data = (vif_data[vif_data['Variable'] != 'const']
            .sort_values('VIF', ascending=False)
            .reset_index(drop=True))
display(vif_data)

print("\nImpacto empírico en el desempeño (Regresión Logística, ROC-AUC CV 5-fold):")

cols_completo = feature_cols
cols_reducido = ['alpha', 'delta', 'r', 'redshift']  # se conserva una sola banda fotométrica (r)

modelo_completo = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=1000, random_state=42))
modelo_reducido = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=1000, random_state=42))

auc_completo = cross_val_score(modelo_completo, X_train[cols_completo], y_train,
                               cv=5, scoring='roc_auc').mean()
auc_reducido = cross_val_score(modelo_reducido, X_train[cols_reducido], y_train,
                               cv=5, scoring='roc_auc').mean()

print(f"  Conjunto completo ({len(cols_completo)} variables): ROC-AUC = {auc_completo:.4f}")
print(f"  Conjunto reducido ({len(cols_reducido)} variables): ROC-AUC = {auc_reducido:.4f}")
print(f"  Diferencia absoluta: {abs(auc_completo - auc_reducido):.4f}")


**Interpretación.** Como era de esperar, las bandas fotométricas `u, g, r, i, z` presentan valores de VIF muy elevados, confirmando una fuerte multicolinealidad entre ellas (son filtros de la misma imagen). Sin embargo, esta redundancia tiene un impacto limitado sobre el problema:

- **Modelos de árboles (Árbol, Random Forest, XGBoost):** son intrínsecamente robustos a la multicolinealidad, ya que seleccionan variables de forma jerárquica; la presencia de predictores correlacionados no degrada su capacidad predictiva. Como son los modelos finalistas, la multicolinealidad no compromete el desempeño.
- **Modelo lineal (Regresión Logística):** la multicolinealidad infla la varianza de los coeficientes y dificulta su interpretación individual, pero la comparación empírica anterior muestra que el ROC-AUC apenas cambia al reducir las variables. Además, la **regularización Ridge** de la Fase 1 mitiga directamente este efecto al penalizar los coeficientes.

**Decisión:** se conservan todas las variables fotométricas. Eliminarlas no se justifica para los modelos de árboles (que dominan el desempeño), y para la regresión logística el problema se controla mediante regularización Ridge en lugar de eliminación de variables.

# FASE 1: LÍNEA BASE CON REGULARIZACIÓN

In [ ]:
# Modelo de Regresión Logística
lr_simple = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_simple.fit(X_train_scaled, y_train)

y_pred_simple = lr_simple.predict(X_test_scaled)
y_pred_proba_simple = lr_simple.predict_proba(X_test_scaled)[:, 1]

print("\nMODELO 1 Regresión Logística Simple")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_simple):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_simple):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_simple):.4f}")

print("\nCoeficientes:")
for var, coef in zip(feature_cols, lr_simple.coef_[0]):
    print(f"  {var:12s}: {coef:8.4f}")

lambda_values = [0.001, 0.01, 0.1, 1, 10, 100]
cv_scores = []

print("\nSELECCIÓN λ Cross-validation 5-fold...")
for lam in lambda_values:
    lr_cv = LogisticRegression(C=1/lam, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)
    scores = cross_val_score(lr_cv, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    cv_scores.append(scores.mean())

optimal_idx = np.argmax(cv_scores)
optimal_lambda = lambda_values[optimal_idx]

print(f"λ óptimo: {optimal_lambda:.6f}, ROC-AUC CV: {cv_scores[optimal_idx]:.4f}")

lr_ridge = LogisticRegression(C=1/optimal_lambda, penalty='l2', solver='lbfgs',
                              max_iter=1000, random_state=42)
lr_ridge.fit(X_train_scaled, y_train)

y_pred_ridge = lr_ridge.predict(X_test_scaled)
y_pred_proba_ridge = lr_ridge.predict_proba(X_test_scaled)[:, 1]

print(f"\nMODELO 2 Regresión Logística Ridge (λ = {optimal_lambda:.2e})")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ridge):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_ridge):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_ridge):.4f}")

In [ ]:
print("\nComparación de coeficientes:")
print(f"{'Variable':<12} {'Simple':>12} {'Ridge':>12} {'% Reducción':>12}")
print("-"*50)
for var, simp, ridge in zip(feature_cols, lr_simple.coef_[0], lr_ridge.coef_[0]):
    pct_red = ((abs(simp) - abs(ridge)) / (abs(simp) + 1e-10)) * 100
    print(f"{var:<12} {simp:12.4f} {ridge:12.4f} {pct_red:11.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sorted_simple = sorted(zip(feature_cols, lr_simple.coef_[0]), key=lambda x: x[1])
var_s, coef_s = zip(*sorted_simple)
axes[0].barh(var_s, coef_s, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Coeficiente')
axes[0].set_title('Regresión Logística Simple')
axes[0].axvline(0, color='black', linewidth=0.8)

sorted_ridge = sorted(zip(feature_cols, lr_ridge.coef_[0]), key=lambda x: x[1])
var_r, coef_r = zip(*sorted_ridge)
axes[1].barh(var_r, coef_r, color='darkgreen', alpha=0.8)
axes[1].set_xlabel('Coeficiente')
axes[1].set_title(f'Ridge (λ={optimal_lambda:.2e})')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

# FASE 2: MODELOS BASADOS EN ÁRBOLES

In [ ]:
# Modelo de Árboles Simple

dt_simple = DecisionTreeClassifier(random_state=42)
dt_simple.fit(X_train_scaled, y_train)
y_pred_proba_dt_simple = dt_simple.predict_proba(X_test_scaled)[:, 1]

print("\nMODELO A Árbol de Decisión Simple")
print(f"Accuracy:  {accuracy_score(y_test, dt_simple.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_dt_simple):.4f}")
print(f"F1-Score:  {f1_score(y_test, dt_simple.predict(X_test_scaled)):.4f}")

print("\nImportancia de variables:")
for var, imp in sorted(zip(feature_cols, dt_simple.feature_importances_),
                       key=lambda x: x[1], reverse=True):
    print(f"  {var:12s}: {imp:.4f}")


fig, ax = plt.subplots(figsize=(16, 10))
plot_tree(dt_simple, feature_names=feature_cols, class_names=['STAR', 'GAL+QSO'],
          filled=True, ax=ax, fontsize=10, max_depth=3)
plt.tight_layout()
plt.show()

# GridSearch para Arboles
print("\nMODELO A - OPTIMIZADO Árbol de Decisión Optimizado (GridSearch)")
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 10, 20, 50],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
dt_grid.fit(X_train_scaled, y_train)

print(f"Mejores parámetros encontrados: {dt_grid.best_params_}")
print(f"Mejor ROC-AUC en CV: {dt_grid.best_score_:.4f}")

dt_grid_tuned = dt_grid.best_estimator_
y_pred_proba_dt_grid_tuned = dt_grid_tuned.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy en Test:  {accuracy_score(y_test, dt_grid_tuned.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC en Test:   {roc_auc_score(y_test, y_pred_proba_dt_grid_tuned):.4f}")
print(f"F1-Score en Test:  {f1_score(y_test, dt_grid_tuned.predict(X_test_scaled)):.4f}")


In [ ]:
# Modelo Random Forest

rf_simple = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_simple.fit(X_train_scaled, y_train)
y_pred_proba_rf_simple = rf_simple.predict_proba(X_test_scaled)[:, 1]

print("\nMODELO B Random Forest")
print(f"Accuracy:  {accuracy_score(y_test, rf_simple.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_rf_simple):.4f}")
print(f"F1-Score:  {f1_score(y_test, rf_simple.predict(X_test_scaled)):.4f}")

print("\nImportancia de variables:")
for var, imp in sorted(zip(feature_cols, rf_simple.feature_importances_),
                       key=lambda x: x[1], reverse=True):
    print(f"  {var:12s}: {imp:.4f}")

# Grid Search para Random Forest
print("\nMODELO B - OPTIMIZADO Random Forest Optimizado (GridSearch)")
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    }

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
rf_grid.fit(X_train_scaled, y_train)

print(f"Mejores parámetros encontrados: {rf_grid.best_params_}")
print(f"Mejor ROC-AUC en CV: {rf_grid.best_score_:.4f}")

rf_tuned = rf_grid.best_estimator_
y_pred_proba_rf_tuned = rf_tuned.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy en Test:  {accuracy_score(y_test, rf_tuned.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC en Test:   {roc_auc_score(y_test, y_pred_proba_rf_tuned):.4f}")
print(f"F1-Score en Test:  {f1_score(y_test, rf_tuned.predict(X_test_scaled)):.4f}")



In [ ]:
# Modelo XGBoost

xgb_simple = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    objective='binary:logistic', eval_metric='auc', seed=42, tree_method='hist'
)
xgb_simple.fit(X_train_scaled, y_train, verbose=0)
y_pred_proba_xgb_simple = xgb_simple.predict_proba(X_test_scaled)[:, 1]

print("\nMODELO C XGBoost")
print(f"Accuracy:  {accuracy_score(y_test, xgb_simple.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb_simple):.4f}")
print(f"F1-Score:  {f1_score(y_test, xgb_simple.predict(X_test_scaled)):.4f}")

# Grid Search para XGBoost
print("\nMODELO C - OPTIMIZADO XGBoost Optimizado (GridSearch)")
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.3],
}

xgb_grid = GridSearchCV(xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc', seed=42, tree_method='hist'),
                        param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train_scaled, y_train)

print(f"Mejores parámetros encontrados: {xgb_grid.best_params_}")
print(f"Mejor ROC-AUC en CV: {xgb_grid.best_score_:.4f}")

xgb_tuned = xgb_grid.best_estimator_
y_pred_proba_xgb_tuned = xgb_tuned.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy en Test:  {accuracy_score(y_test, xgb_tuned.predict(X_test_scaled)):.4f}")
print(f"ROC-AUC en Test:   {roc_auc_score(y_test, y_pred_proba_xgb_tuned):.4f}")
print(f"F1-Score en Test:  {f1_score(y_test, xgb_tuned.predict(X_test_scaled)):.4f}")


# FASE 2.5: ENSAMBLAJE DE MODELOS

Se construye un modelo de **ensamblaje por votación suave (soft voting)** que combina las mejores versiones de cada familia entrenada (Regresión Logística Ridge, Árbol de Decisión optimizado, Random Forest optimizado y XGBoost optimizado). El voto suave promedia las probabilidades predichas por cada modelo, de modo que el ensamblaje puede compensar errores individuales. Después se compara si conviene implementar este ensamblaje o el mejor modelo individual.

In [ ]:
# Ensamblaje por votación suave (promedio de probabilidades)
estimators = [
    ('lr',  lr_ridge),
    ('dt',  dt_grid_tuned),
    ('rf',  rf_tuned),
    ('xgb', xgb_tuned),
]

ensemble_soft = VotingClassifier(estimators=estimators, voting='soft', n_jobs=-1)
ensemble_soft.fit(X_train_scaled, y_train)

y_pred_ensemble = ensemble_soft.predict(X_test_scaled)
y_pred_proba_ensemble = ensemble_soft.predict_proba(X_test_scaled)[:, 1]

print("MODELO D - ENSAMBLAJE (Voting Soft: LR Ridge + Árbol + RF + XGB)")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ensemble):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_ensemble):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_ensemble):.4f}")

# Registro completo de modelos (para comparación y selección posterior)
registro_modelos = {
    'Logística Simple':              (lr_simple,     y_pred_proba_simple),
    'Ridge':                         (lr_ridge,      y_pred_proba_ridge),
    'Árbol Simple':                  (dt_simple,     y_pred_proba_dt_simple),
    'Árbol Optimizado (GridSearch)': (dt_grid_tuned, y_pred_proba_dt_grid_tuned),
    'RF Simple':                     (rf_simple,     y_pred_proba_rf_simple),
    'RF Optimizado (GridSearch)':    (rf_tuned,      y_pred_proba_rf_tuned),
    'XGB Simple':                    (xgb_simple,    y_pred_proba_xgb_simple),
    'XGB Optimizado (GridSearch)':   (xgb_tuned,     y_pred_proba_xgb_tuned),
    'Ensamblaje (Voting Soft)':      (ensemble_soft, y_pred_proba_ensemble),
}


In [ ]:
results_data = {
    'Modelo': [
        'Logística Simple', 'Ridge',
        'Árbol Simple', 'Árbol Optimizado (GridSearch)',
        'RF Simple', 'RF Optimizado (GridSearch)',
        'XGB Simple', 'XGB Optimizado (GridSearch)',
        'Ensamblaje (Voting Soft)'
    ],
    'Accuracy': [
        accuracy_score(y_test, lr_simple.predict(X_test_scaled)),
        accuracy_score(y_test, lr_ridge.predict(X_test_scaled)),
        accuracy_score(y_test, dt_simple.predict(X_test_scaled)),
        accuracy_score(y_test, dt_grid_tuned.predict(X_test_scaled)),
        accuracy_score(y_test, rf_simple.predict(X_test_scaled)),
        accuracy_score(y_test, rf_tuned.predict(X_test_scaled)),
        accuracy_score(y_test, xgb_simple.predict(X_test_scaled)),
        accuracy_score(y_test, xgb_tuned.predict(X_test_scaled)),
        accuracy_score(y_test, y_pred_ensemble)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_proba_simple),
        roc_auc_score(y_test, y_pred_proba_ridge),
        roc_auc_score(y_test, y_pred_proba_dt_simple),
        roc_auc_score(y_test, y_pred_proba_dt_grid_tuned),
        roc_auc_score(y_test, y_pred_proba_rf_simple),
        roc_auc_score(y_test, y_pred_proba_rf_tuned),
        roc_auc_score(y_test, y_pred_proba_xgb_simple),
        roc_auc_score(y_test, y_pred_proba_xgb_tuned),
        roc_auc_score(y_test, y_pred_proba_ensemble)
    ],
    'F1-Score': [
        f1_score(y_test, lr_simple.predict(X_test_scaled)),
        f1_score(y_test, lr_ridge.predict(X_test_scaled)),
        f1_score(y_test, dt_simple.predict(X_test_scaled)),
        f1_score(y_test, dt_grid_tuned.predict(X_test_scaled)),
        f1_score(y_test, rf_simple.predict(X_test_scaled)),
        f1_score(y_test, rf_tuned.predict(X_test_scaled)),
        f1_score(y_test, xgb_simple.predict(X_test_scaled)),
        f1_score(y_test, xgb_tuned.predict(X_test_scaled)),
        f1_score(y_test, y_pred_ensemble)
    ]
}

results_df = pd.DataFrame(results_data)
results_df.set_index('Modelo', inplace=True)
results_df


In [ ]:
print("RESUMEN COMPARATIVO")
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x_pos = np.arange(len(results_df))
axes[0].bar(x_pos, results_df['Accuracy'], alpha=0.8, color='steelblue')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Comparación de Accuracy')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_df.index, rotation=45, ha='right', fontsize=9)
axes[0].set_ylim([0.7, 1.0])
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x_pos, results_df['ROC-AUC'], alpha=0.8, color='darkgreen')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Comparación de ROC-AUC')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(results_df.index, rotation=45, ha='right', fontsize=9)
axes[1].set_ylim([0.7, 1.0])
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(x_pos, results_df['F1-Score'], alpha=0.8, color='darkred')
axes[2].set_ylabel('F1-Score')
axes[2].set_title('Comparación de F1-Score')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(results_df.index, rotation=45, ha='right', fontsize=9)
axes[2].set_ylim([0.7, 1.0])
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

Los resultados confirman que los métodos de ensamble basados en árboles (Random Forest y XGBoost) son los más adecuados para separar objetos estelares (STAR) de objetos extragalácticos (GALAXY + QSO) a partir de los datos del SDSS, con valores de ROC-AUC y exactitud muy altos. Como ahora la muestra está **balanceada (50% STAR / 50% GAL+QSO)**, estas métricas no están infladas por el desbalance de clases y reflejan de forma más fiable la capacidad real de discriminación.

**¿Ensamblaje o modelo individual?** El ensamblaje por votación suave combina las cuatro familias de modelos y suele aportar algo de estabilidad. Conviene comparar su ganancia real frente al mejor modelo individual (XGBoost optimizado):

- Si el ensamblaje mejora el ROC-AUC/F1 de forma marginal (p. ej. < 0.005), **no compensa** su mayor costo computacional y de mantenimiento (entrena y sirve cuatro modelos). En ese caso es preferible implementar **XGBoost optimizado** por su simplicidad, velocidad de inferencia e interpretabilidad vía SHAP.
- Si el ensamblaje aporta una mejora consistente y relevante, puede justificarse su implementación cuando la prioridad sea maximizar la robustez.

**Recomendación final:** para este conjunto de datos, salvo que el ensamblaje muestre una mejora clara en la tabla anterior, se recomienda implementar **XGBoost optimizado**, por ofrecer el mejor equilibrio entre precisión, generalización, velocidad e interpretabilidad. El ensamblaje queda como alternativa cuando se priorice la estabilidad por encima del costo operativo.

## Análisis de la Matriz de Confusión para el Modelo Ganador

Se analiza la matriz de confusión del modelo **XGBoost Optimizado (GridSearch)**. Recuerde que el problema es binario tras agrupar GALAXY y QSO en una sola categoría:

- **0 = STAR** (objeto estelar)
- **1 = GAL+QSO** (objeto extragaláctico)

In [ ]:
# Predicciones del modelo ganador (XGBoost Optimizado)
y_pred_xgb_tuned = xgb_tuned.predict(X_test_scaled)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_xgb_tuned)

etiquetas = ['STAR (0)', 'GAL+QSO (1)']
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=etiquetas, yticklabels=etiquetas)
plt.title('Matriz de Confusión - XGBoost (GridSearch)')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

# y=1 -> GAL+QSO (clase positiva); y=0 -> STAR (clase negativa)
tn, fp, fn, tp = cm.ravel()

print("\nResultados de la Matriz de Confusión para XGBoost:")
print(f"  TP (GAL+QSO clasificados correctamente):           {tp}")
print(f"  TN (STAR clasificadas correctamente):              {tn}")
print(f"  FP (STAR clasificadas erróneamente como GAL+QSO):  {fp}")
print(f"  FN (GAL+QSO clasificados erróneamente como STAR):  {fn}")

accuracy_cm = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nExactitud (Accuracy):   {accuracy_cm:.4f}")
print(f"Precisión (Precision):  {precision:.4f}")
print(f"Sensibilidad (Recall):  {recall:.4f}")
print(f"Puntuación F1:          {f1:.4f}")


# FASE 3: XAI - EXPLICABILIDAD

In [ ]:
# Inicializar el explicador SHAP para XGBoost
# El modelo predice la clase positiva (1 = GAL+QSO / extragaláctico).
# Valores SHAP positivos empujan la predicción hacia GAL+QSO; negativos hacia STAR.
explainer = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_test_scaled)

# Gráfico resumen SHAP
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols)


El gráfico SHAP confirma que el **redshift** es, con diferencia, la variable más influyente del modelo XGBoost. Esto es coherente con la agrupación realizada: tanto las galaxias como los cuásares (QSO) son objetos extragalácticos con redshift positivo, mientras que las estrellas (STAR) tienen un redshift prácticamente nulo. Por ello, valores altos de redshift empujan con fuerza la predicción hacia la clase positiva (GAL+QSO), seguidos por los filtros fotométricos `g, z, u, i`.

Los pocos errores se concentran en la zona de solapamiento:

- **Falsos positivos (STAR predicha como GAL+QSO):** estrellas con un redshift medido ligeramente anómalo o con una combinación de filtros atípica que imita a un objeto extragaláctico.
- **Falsos negativos (GAL+QSO predicho como STAR):** galaxias muy cercanas con redshift casi nulo, donde el modelo pierde su discriminador principal y debe apoyarse en los filtros, que en estos casos se asemejan a los de una estrella.

Los conteos exactos de TP, TN, FP y FN corresponden a la matriz de confusión mostrada arriba. En conjunto, el redshift actúa como discriminador dominante y los filtros fotométricos como respaldo.

In [ ]:
print(f"\n[LIME] Configurando explicador para XGBoost...")

# Usar xgb_tuned como el modelo para la explicación LIME
current_model = xgb_tuned
current_proba = y_pred_proba_xgb_tuned

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train_scaled,
    feature_names=feature_cols,
    class_names=['STAR', 'GAL+QSO'],  # 0 = STAR, 1 = GAL+QSO (debe coincidir con y)
    mode='classification',
    random_state=42
)

errors = np.abs(y_test.values - (current_proba > 0.5).astype(int))
error_indices = np.where(errors == 1)[0]
correct_indices = np.where(errors == 0)[0]

if len(correct_indices) > 0:
    # Encontrar el índice de una predicción correcta cercana al umbral de decisión (0.5)
    success_idx = correct_indices[np.argmin(np.abs(current_proba[correct_indices] - 0.5))]
elif len(y_test) > 0:
    success_idx = correct_indices[0]
else:
    success_idx = 0 # Fallback si no hay predicciones correctas (poco probable)

if len(error_indices) > 0:
    error_idx = error_indices[0]
else:
    error_idx = None

print(f"\n[CASO 1] ÉXITO (índice {success_idx})")
print(f"Predicción: {current_proba[success_idx]:.4f}")
print(f"Clase real: {'GAL+QSO' if y_test.iloc[success_idx] == 1 else 'STAR'}")

exp_success = explainer_lime.explain_instance(
    X_test_scaled[success_idx],
    current_model.predict_proba,
    num_features=len(feature_cols)
)

print("Variables influyentes:")
for feature, weight in exp_success.as_list():
    print(f"  {feature}: {weight:+.4f}")

# Visualización LIME para predicción correcta
exp_success.show_in_notebook(show_table=True, show_all=False)
fig_success = exp_success.as_pyplot_figure()
fig_success.suptitle(f'LIME Explanation for Correct Prediction (Index {success_idx})', y=1.02)
plt.tight_layout()
plt.show()

if error_idx is not None:
    print(f"\n[CASO 2] ERROR (índice {error_idx})")
    print(f"Predicción: {current_proba[error_idx]:.4f}")
    real = 'GAL+QSO' if y_test.iloc[error_idx] == 1 else 'STAR'
    pred = 'GAL+QSO' if current_proba[error_idx] > 0.5 else 'STAR'
    print(f"Real: {real}, Predicho: {pred}")

    exp_error = explainer_lime.explain_instance(
        X_test_scaled[error_idx],
        current_model.predict_proba,
        num_features=len(feature_cols)
    )

    print("Variables influyentes:")
    for feature, weight in exp_error.as_list():
        print(f"  {feature}: {weight:+.4f}")

    # Visualización LIME para error
    exp_error.show_in_notebook(show_table=True, show_all=False)
    fig_error = exp_error.as_pyplot_figure()
    fig_error.suptitle(f'LIME Explanation for Error (Index {error_idx})', y=1.02)
    plt.tight_layout()
    plt.show()

# Analizar falsos positivos y falsos negativos usando el modelo XGBoost
confusion = confusion_matrix(y_test, current_proba > 0.5)
tn, fp, fn, tp = confusion.ravel()

# Encontrar y explicar un Falso Positivo (Star clasificada como Galaxy)
if fp > 0:
    fp_mask = (y_test.values == 0) & ((current_proba > 0.5).astype(int) == 1)
    # Tomar el primer índice de falso positivo
    fp_idx = np.where(fp_mask)[0][0]
    print(f"\n[CASO 3] FALSO POSITIVO (índice {fp_idx})")
    print(f"Predicción: {current_proba[fp_idx]:.4f} → GAL+QSO")
    print(f"Clase real: STAR")

    exp_fp = explainer_lime.explain_instance(
        X_test_scaled[fp_idx],
        current_model.predict_proba,
        num_features=len(feature_cols)
    )
    print("Variables influyentes:")
    for feature, weight in exp_fp.as_list():
        print(f"  {feature}: {weight:+.4f}")

    # Visualización LIME para falso positivo
    exp_fp.show_in_notebook(show_table=True, show_all=False)
    fig_fp = exp_fp.as_pyplot_figure()
    fig_fp.suptitle(f'LIME Explanation for False Positive (Index {fp_idx})', y=1.02)
    plt.tight_layout()
    plt.show()

# Encontrar y explicar un Falso Negativo (Galaxy clasificada como Star)
if fn > 0:
    fn_mask = (y_test.values == 1) & ((current_proba > 0.5).astype(int) == 0)
    # Tomar el primer índice de falso negativo
    fn_idx = np.where(fn_mask)[0][0]
    print(f"\n[CASO 4] FALSO NEGATIVO (índice {fn_idx})")
    print(f"Predicción: {current_proba[fn_idx]:.4f} → STAR")
    print(f"Clase real: GAL+QSO")

    exp_fn = explainer_lime.explain_instance(
        X_test_scaled[fn_idx],
        current_model.predict_proba,
        num_features=len(feature_cols)
    )
    print("Variables influyentes:")
    for feature, weight in exp_fn.as_list():
        print(f"  {feature}: {weight:+.4f}")

    # Visualización LIME para falso negativo
    exp_fn.show_in_notebook(show_table=True, show_all=False)
    fig_fn = exp_fn.as_pyplot_figure()
    fig_fn.suptitle(f'LIME Explanation for False Negative (Index {fn_idx})', y=1.02)
    plt.tight_layout()
    plt.show()

## Interpretación del Análisis LIME para XGBoost

El análisis LIME ofrece una visión local de cómo el modelo XGBoost decide para instancias individuales, complementando el análisis global de SHAP. Recuerde que la clase positiva es `GAL+QSO` (extragaláctico) y la negativa es `STAR`.

### Caso 1: Predicción Correcta (Éxito)
Para una instancia clasificada correctamente (p. ej. un objeto extragaláctico predicho como `GAL+QSO`), LIME suele mostrar que las variables que impulsan la predicción son coherentes con la física: un `redshift` elevado y combinaciones de filtros fotométricos típicas de galaxias/cuásares contribuyen positivamente. Esto valida que el modelo aprende patrones físicamente significativos.

### Caso 2: Error de Predicción (General)
Cuando el modelo se equivoca, LIME ayuda a entender por qué. Por ejemplo, si una `STAR` real se predice como `GAL+QSO`, LIME puede revelar que su `redshift`, aunque bajo, resultó algo elevado respecto a otras estrellas, o que sus filtros se asemejaban a los de un objeto extragaláctico.

### Caso 3: Falso Positivo (STAR clasificada como GAL+QSO)
En un falso positivo, una `STAR` se clasifica erróneamente como `GAL+QSO`. LIME suele resaltar que un `redshift` ligeramente anómalo, o una configuración inusual de los filtros `u, g, r, i, z` que se solapa con las características extragalácticas, lleva al modelo a sobrestimar la probabilidad de la clase positiva.

### Caso 4: Falso Negativo (GAL+QSO clasificado como STAR)
Para un falso negativo, un objeto extragaláctico real se clasifica como `STAR`. LIME suele mostrar que su `redshift` es inusualmente bajo (similar al de una estrella) y que sus filtros no exhiben las firmas distintivas esperadas. Esto es común en galaxias muy cercanas, donde el redshift cosmológico es casi indetectable y el aspecto fotométrico resulta más ambiguo para el modelo.

In [ ]:
# Selección automática de los DOS mejores modelos con importancia de variables
# (basada en ROC-AUC). Se restringe a modelos basados en árboles, que exponen
# feature_importances_ y dominan el desempeño.
modelos_con_importancia = {
    nombre: modelo
    for nombre, (modelo, _) in registro_modelos.items()
    if hasattr(modelo, 'feature_importances_')
}
ranking_imp = sorted(
    modelos_con_importancia.items(),
    key=lambda kv: roc_auc_score(y_test, registro_modelos[kv[0]][1]),
    reverse=True
)
model_1_name, best_model = ranking_imp[0]
model_2_name, second_model = ranking_imp[1]
print(f"Mejor modelo (importancia): {model_1_name}")
print(f"Segundo modelo (importancia): {model_2_name}\n")

print("IMPORTANCIA DE VARIABLES")

print(f"\n[{model_1_name}]")
if hasattr(best_model, 'feature_importances_'):
    for var, imp in sorted(zip(feature_cols, best_model.feature_importances_),
                          key=lambda x: x[1], reverse=True):
        print(f"  {var:12s}: {imp:.4f}")

print(f"\n[{model_2_name}]")
if hasattr(second_model, 'feature_importances_'):
    for var, imp in sorted(zip(feature_cols, second_model.feature_importances_),
                          key=lambda x: x[1], reverse=True):
        print(f"  {var:12s}: {imp:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if hasattr(best_model, 'feature_importances_'):
    vars_sorted = sorted(zip(feature_cols, best_model.feature_importances_), key=lambda x: x[1])
    var_1, imp_1 = zip(*vars_sorted)
    axes[0].barh(var_1, imp_1, alpha=0.8, color='steelblue')
    axes[0].set_xlabel('Importancia')
    axes[0].set_title(f'Feature Importance - {model_1_name}')
    axes[0].grid(axis='x', alpha=0.3)

if hasattr(second_model, 'feature_importances_'):
    vars_sorted = sorted(zip(feature_cols, second_model.feature_importances_), key=lambda x: x[1])
    var_2, imp_2 = zip(*vars_sorted)
    axes[1].barh(var_2, imp_2, alpha=0.8, color='darkgreen')
    axes[1].set_xlabel('Importancia')
    axes[1].set_title(f'Feature Importance - {model_2_name}')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Sentido lógico de las variables y su importancia

Desde una perspectiva física, las magnitudes fotométricas en las bandas u, g, r, i y z representan el brillo de los objetos medido en distintas regiones del espectro electromagnético. Estas bandas resultan altamente discriminativas porque los objetos estelares y extragalácticos presentan firmas espectrales diferenciadas. El desplazamiento al rojo o redshift constituye la variable de mayor poder discriminativo: las estrellas, al ser objetos cercanos dentro de nuestra galaxia, tienen un valor prácticamente nulo, mientras que **tanto las galaxias como los cuásares (QSO) muestran valores positivos** debido a la expansión del universo. Precisamente esta propiedad común justifica agrupar GALAXY y QSO en una única categoría extragaláctica. En cambio, las coordenadas angulares (ascensión recta y declinación) aportan una capacidad de discriminación muy baja. En conjunto, el comportamiento observado en los modelos resulta plenamente consistente con los fundamentos de la física observacional.